In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load S&P 500 Data
# sp500_data = pd.read_csv('../../data/final/sp500_monthly_data.csv')
sp500_data = pd.read_csv('../../data/yfinance/sp500_yfinance_monthly.csv')
sp500_data_edgar = pd.read_csv('../../data/edgar_sec/sp500_edgar_sec.csv')

# Load Gig Economy Data
gig_data = pd.read_csv('../../data/yfinance/gig_yfinance_monthly.csv')
gig_data_edgar = pd.read_csv('../../data/edgar_sec/gig_edgar_sec.csv')

def yfin_merge_prep(df):
    df.columns = df.columns.str.capitalize()
    df = df.dropna(subset=['Close'])
    df = df.loc[:, ~df.columns.duplicated()]
    return df

# all columns capitalized
sp500_data = yfin_merge_prep(sp500_data)
gig_data = yfin_merge_prep(gig_data)

def edgar_merge_prep(df):
    df.rename(columns={'date': 'Date', 'ticker': 'Ticker'}, inplace=True)
    df = df.dropna(subset=['NetIncomeLoss', 'NetWorth', 'EarningsPerShareDiluted', 'SharesOutstanding'])
    df = df[['Date', 'Ticker', 'NetIncomeLoss', 'NetWorth', 'EarningsPerShareDiluted', 'SharesOutstanding']].copy()
    return df

sp500_data_edgar = edgar_merge_prep(sp500_data_edgar)
gig_data_edgar = edgar_merge_prep(gig_data_edgar)

def merge_yfin_edgar(yfin_df, edgar_df):
    merged_df = pd.merge(yfin_df, edgar_df, on=['Date', 'Ticker'], how='left', suffixes=('', '_edgar'))
    merged_df.sort_values(by=['Ticker', 'Date'], inplace=True)
    edgar_cols = ['NetIncomeLoss', 'NetWorth', 'EarningsPerShareDiluted', 'SharesOutstanding']
    merged_df[edgar_cols] = merged_df.groupby('Ticker')[edgar_cols].transform(lambda x: x.bfill().ffill())
    return merged_df

gig_merged = merge_yfin_edgar(gig_data, gig_data_edgar)
sp500_merged = merge_yfin_edgar(sp500_data, sp500_data_edgar)

sp500_data = sp500_merged.copy().dropna()
gig_data = gig_merged.copy().dropna()

# Display shapes and date ranges
def data_summary(df, name):
    return {
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Date Min': df['Date'].min(),
        'Date Max': df['Date'].max(),
        'Companies': df['Ticker'].nunique(),
        'Missing Values': df.isnull().sum().sum()
    }

summary_df = pd.DataFrame([
    data_summary(sp500_data, 'S&P 500'),
    data_summary(gig_data, 'Gig Economy')
])

summary_df

,Dataset,Rows,Columns,Date Min,Date Max,Companies,Missing Values
0,S&P 500,94451,14,2009-01-31,2025-12-31,495,0
1,Gig Economy,1531,14,2009-01-31,2025-12-31,13,0


In [3]:
def merge_sp500_gig(sp500_data, gig_data):
    # Add category labels
    sp500_data['Category'] = 'sp500'
    gig_data['Category'] = 'gig-work'
    # 'HLF', 'NUS',  'PRI', 'USNA' to mlm-work Category
    mlm_tickers = ['HLF', 'NUS', 'PRI', 'USNA']
    gig_data.loc[gig_data['Ticker'].isin(mlm_tickers), 'Category'] = 'mlm-work'

    # if Ticker exists in both datsets, removee the duplicate from S&P 500 data
    overlap_tickers = set(sp500_data['Ticker']).intersection(set(gig_data['Ticker']))
    print(f"Overlapping Tickers: {overlap_tickers}")
    sp500_data = sp500_data[~sp500_data['Ticker'].isin(overlap_tickers)]
    
    gig_tickers = gig_data[gig_data['Category'] == 'gig-work']['Ticker'].unique()
    print(f"Gig Economy Tickers in Cleaned Data: {gig_tickers}")
    mlm_tickers_in_data = gig_data[gig_data['Category'] == 'mlm-work']['Ticker'].unique()
    print(f"MLM Tickers in Cleaned Data: {mlm_tickers_in_data}")

    # Combine datasets
    combined_data = pd.concat([sp500_data, gig_data], ignore_index=True)

    # Calculate Market Cap
    combined_data['MarketCap'] = combined_data['Close'] * combined_data['SharesOutstanding']

    combined_data = combined_data.drop(columns=[
        # 'Industrykey', 
        'Sectorkey',
    ], errors='ignore')
    return combined_data

def input_start_date(df):
    # Group by Ticker and determine earliest data point(start date) for each company
    earliest_start_dates = df.groupby('Ticker')['Date'].min().reset_index()
    earliest_start_dates.columns = ['Ticker', 'EarliestStartDate']
    earliest_start_dates = earliest_start_dates.sort_values(by='EarliestStartDate', ascending=True)

    earliest_start_dates['EarliestStartDate'] = pd.to_datetime(earliest_start_dates['EarliestStartDate'])

    # Use start date near 90.5th percentile of all earliest start dates as data range cutoff(captures approximately 5 years before and after the start of COVID)
    start_date_to_use = earliest_start_dates['EarliestStartDate'].iloc[int(len(earliest_start_dates) * 0.905)]  # 90.5th percentile
    print(f"Start Date to Use: {start_date_to_use.date()}")
    input_data_filtered = df[df['Date'] >= str(start_date_to_use)].copy()
    print(f"\nNumber of Companies after cleaning: {input_data_filtered['Ticker'].nunique()}")
    
    input_data_filtered.reset_index(drop=True, inplace=True)
    
    return input_data_filtered

def resample_quarterly(df):
    # condense to quartely data prior to calculating changes and rolling stats since financials are quarterly()
    df['date'] = pd.to_datetime(df['Date'])
    df['ticker'] = df['Ticker']
    df = df.sort_values(by=['ticker', 'date'])
    df = df.groupby(['ticker']).resample('QE', on='date').last().reset_index(drop=True)

    df.isna().sum()
    
    return df

def feature_engineering(df):

    # Convert Date to datetime
    df['Date'] = pd.to_datetime(df['Date'])

    # Calculate returns and log returns
    df = df.sort_values(['Ticker', 'Date'])
    df['Return'] = df.groupby('Ticker')['Close'].pct_change()
    df['Log_Return'] = df.groupby('Ticker')['Close'].transform(lambda x: np.log(x / x.shift(1)))

    # Volume changes
    df['Volume_Change'] = df.groupby('Ticker')['Volume_accum'].pct_change()

    # NetIncomeLoss, NetWorthm and EarningsPerShareDiluted (3-month pct changes since financials are quarterly)
    df['NIL_Change'] = df.groupby('Ticker')['NetIncomeLoss'].pct_change()
    df['NetWorth_Change'] = df.groupby('Ticker')['NetWorth'].pct_change()
    df['EPS_Change'] = df.groupby('Ticker')['EarningsPerShareDiluted'].pct_change()

    # Rolling statistics (2 & 4-quarter window)
    for ticker in df['Ticker'].unique():
        mask = df['Ticker'] == ticker
        df.loc[mask, 'Volatility_2Q'] = df.loc[mask, 'Return'].rolling(window=2).std()
        df.loc[mask, 'Avg_Return_2Q'] = df.loc[mask, 'Return'].rolling(window=2).mean()
        df.loc[mask, 'Avg_Volume_Change_2Q'] = df.loc[mask, 'Volume_Change'].rolling(window=2).mean()
        df.loc[mask, 'NIL_Volatility_2Q'] = df.loc[mask, 'NIL_Change'].rolling(window=2).std()
        df.loc[mask, 'Avg_NIL_Change_2Q'] = df.loc[mask, 'NIL_Change'].rolling(window=2).mean()
        df.loc[mask, 'NetWorth_Volatility_2Q'] = df.loc[mask, 'NetWorth_Change'].rolling(window=2).std()
        df.loc[mask, 'Avg_NetWorth_Change_2Q'] = df.loc[mask, 'NetWorth_Change'].rolling(window=2).mean()
        df.loc[mask, 'EPS_Volatility_2Q'] = df.loc[mask, 'EPS_Change'].rolling(window=2).std()
        df.loc[mask, 'Avg_EPS_Change_2Q'] = df.loc[mask, 'EPS_Change'].rolling(window=2).mean()
        df.loc[mask, 'Volatility_4Q'] = df.loc[mask, 'Return'].rolling(window=4).std()
        df.loc[mask, 'Avg_Return_4Q'] = df.loc[mask, 'Return'].rolling(window=4).mean()
        df.loc[mask, 'Avg_Volume_Change_4Q'] = df.loc[mask, 'Volume_Change'].rolling(window=4).mean()
        df.loc[mask, 'NIL_Volatility_4Q'] = df.loc[mask, 'NIL_Change'].rolling(window=4).std()
        df.loc[mask, 'Avg_NIL_Change_4Q'] = df.loc[mask, 'NIL_Change'].rolling(window=4).mean()
        df.loc[mask, 'NetWorth_Volatility_4Q'] = df.loc[mask, 'NetWorth_Change'].rolling(window=4).std()
        df.loc[mask, 'Avg_NetWorth_Change_4Q'] = df.loc[mask, 'NetWorth_Change'].rolling(window=4).mean()
        df.loc[mask, 'EPS_Volatility_4Q'] = df.loc[mask, 'EPS_Change'].rolling(window=4).std()
        df.loc[mask, 'Avg_EPS_Change_4Q'] = df.loc[mask, 'EPS_Change'].rolling(window=4).mean()

    return df

input_data = merge_sp500_gig(sp500_data, gig_data)
input_data_filtered = input_start_date(input_data)
input_data_quarterly = resample_quarterly(input_data_filtered)
input_df = feature_engineering(input_data_quarterly).dropna()
input_df.head()

Overlapping Tickers: {'DASH', 'UBER'}
Gig Economy Tickers in Cleaned Data: <ArrowStringArray>
['CART', 'DASH', 'ETSY', 'FVRR', 'LYFT', 'SHOP', 'UBER', 'UDMY', 'UPWK']
Length: 9, dtype: str
MLM Tickers in Cleaned Data: <ArrowStringArray>
['HLF', 'NUS', 'PRI', 'USNA']
Length: 4, dtype: str
Start Date to Use: 2014-10-31

Number of Companies after cleaning: 506


,Date,Month,Quarter,Year,Company,Ticker,Industrykey,Close,Volume_accum,NetIncomeLoss,...,Avg_EPS_Change_2Q,Volatility_4Q,Avg_Return_4Q,Avg_Volume_Change_4Q,NIL_Volatility_4Q,Avg_NIL_Change_4Q,NetWorth_Volatility_4Q,Avg_NetWorth_Change_4Q,EPS_Volatility_4Q,Avg_EPS_Change_4Q
4,2015-12-31,12,4,2015,Agilent Technologies,A,diagnostics-research,38.515442,59284400.0,3.894000e+09,...,0.467172,0.146801,0.016093,0.072934,9.427089,-5.238005,0.001074,0.000709,0.694633,0.204653
5,2016-03-31,3,1,2016,Agilent Technologies,A,diagnostics-research,36.709888,37122800.0,4.072870e+09,...,-0.205399,0.149987,-0.000599,-0.000458,9.471305,-5.176937,0.106995,-0.053309,0.738134,0.182610
6,2016-06-30,6,2,2016,Agilent Technologies,A,diagnostics-research,41.090763,52011000.0,2.820251e+09,...,0.228811,0.150876,0.046524,0.021100,9.610725,-4.944259,0.106995,-0.053309,0.940991,0.347992
7,2016-09-30,9,3,2016,Agilent Technologies,A,diagnostics-research,43.727928,38864400.0,7.085573e+07,...,1.050877,0.111773,0.089463,0.008203,0.442111,-0.348301,0.106995,-0.053309,0.961782,0.422739
8,2016-12-31,12,4,2016,Agilent Technologies,A,diagnostics-research,42.429661,34086300.0,4.048000e+09,...,0.905128,0.078698,0.026737,-0.087119,28.274336,13.723420,0.106900,-0.053450,1.018799,0.566970


In [4]:
def industry_distribution(df):
    # distribution to tickers per industry
    cat_counts = df.groupby('Industrykey')['Ticker'].nunique().reset_index(name='CompanyCount')
    for cat in cat_counts['Industrykey']:
        count = cat_counts[cat_counts['Industrykey'] == cat]['CompanyCount'].values[0]
        print(f"{cat}: {count} companies")
    average_companies_per_industry = cat_counts['CompanyCount'].mean()
    print(f"\nAverage number of companies per industry: {average_companies_per_industry:.2f}")

industry_distribution(input_df)

advertising-agencies: 3 companies
aerospace-defense: 12 companies
agricultural-inputs: 3 companies
airlines: 3 companies
apparel-manufacturing: 1 companies
apparel-retail: 3 companies
asset-management: 13 companies
auto-manufacturers: 3 companies
auto-parts: 4 companies
auto-truck-dealerships: 1 companies
banks-diversified: 5 companies
banks-regional: 8 companies
beverages-brewers: 1 companies
beverages-non-alcoholic: 4 companies
biotechnology: 5 companies
building-materials: 3 companies
building-products-equipment: 6 companies
capital-markets: 5 companies
chemicals: 1 companies
communication-equipment: 6 companies
computer-hardware: 6 companies
confectioners: 2 companies
conglomerates: 2 companies
consulting-services: 2 companies
consumer-electronics: 1 companies
copper: 1 companies
credit-services: 5 companies
diagnostics-research: 11 companies
discount-stores: 5 companies
drug-manufacturers-general: 9 companies
drug-manufacturers-specialty-generic: 2 companies
education-training-ser

In [5]:
def aggregation(df, label=None):
    # Aggregate metrics by ticker for clustering
    engineered_features = df.groupby(['Ticker', 'Company', 'Industrykey', 'Category']).agg({
        # 'Close': ['mean', 'std', 'min', 'max'],
        'Return': ['mean', 'std', 'skew'],
        'Volume_accum': ['mean', 'std', 'max'],
        # 'SharesOutstanding': ['mean', 'std'],
        'MarketCap': ['mean', 'std', 'min', 'max'],
        'Volume_Change': ['mean', 'std'],
        'EPS_Change': ['mean', 'std'],
        'NetWorth_Change': ['mean', 'std'],
        'NIL_Change': ['mean', 'std'],
        # 'Log_Return': ['mean', 'std', 'skew'],
        'Volatility_2Q': ['mean', 'max'],
        'Avg_Return_2Q': 'mean',
        'Avg_Volume_Change_2Q': 'mean',
        'EPS_Volatility_2Q': ['mean', 'max'],
        'Avg_EPS_Change_2Q': 'mean',
        'NetWorth_Volatility_2Q': ['mean', 'max'],
        'Avg_NetWorth_Change_2Q': 'mean',
        'NIL_Volatility_2Q': ['mean', 'max'],
        'Avg_NIL_Change_2Q': 'mean',
        'Volatility_4Q': ['mean', 'max'],
        'Avg_Return_4Q': 'mean',
        'Avg_Volume_Change_4Q': 'mean',
        'EPS_Volatility_4Q': ['mean', 'max'],
        'Avg_EPS_Change_4Q': 'mean',
        'NetWorth_Volatility_4Q': ['mean', 'max'],
        'Avg_NetWorth_Change_4Q': 'mean',
        'NIL_Volatility_4Q': ['mean', 'max'],
        'Avg_NIL_Change_4Q': 'mean'
    }).reset_index()

    # Flatten column names
    engineered_features.columns = ['_'.join(col).strip('_') for col in engineered_features.columns.values]
    engineered_features.rename(columns={'Ticker_': 'Ticker', 'Company_': 'Company', 'Industrykey_': 'Industrykey', 'Category_': 'Category'}, inplace=True)
    eng_feat_non_numeric = ['Ticker', 'Company', 'Industrykey', 'Category']

    # Skew can be undefined for constant or too-small groups; treat those as no skew.
    skew_cols = [col for col in engineered_features.columns if col.endswith('_skew')]
    if skew_cols:
        engineered_features[skew_cols] = engineered_features[skew_cols].fillna(0.0)

    # Clean any remaining invalid values before scaling
    engineered_features = engineered_features.replace([np.inf, -np.inf], np.nan).dropna()

    # Select numeric features for clustering
    feature_cols = [col for col in engineered_features.columns if col not in eng_feat_non_numeric]

    tag = f" [{label}]" if label else ""
    print(f"\nTicker-level features shape{tag}: {engineered_features.shape}")
    print(f"\nFeatures:\n{engineered_features.columns.tolist()}")

    return engineered_features, feature_cols, eng_feat_non_numeric


def aggregation_by_period(df):
    date_col = pd.to_datetime(df['Date'])

    period_masks = {
        'all':        slice(None),                                          # all rows
        'pre_covid':  date_col <= '2019-12-31',
        'covid':     (date_col >= '2020-01-01') & (date_col <= '2022-12-31'),
        'post_covid': date_col >= '2023-01-01',
    }

    results = {}
    for period, mask in period_masks.items():
        subset = df.loc[mask].copy() if not isinstance(mask, slice) else df.copy()
        results[period] = aggregation(subset, label=period)

    return results


period_aggregations = aggregation_by_period(input_df)
pre_covid_ef, pre_covid_fc, pre_covid_en = period_aggregations['pre_covid']
covid_ef, covid_fc, covid_en = period_aggregations['covid']
post_covid_ef, post_covid_fc, post_covid_en = period_aggregations['post_covid']

engineered_features, feature_cols, eng_feat_non_numeric = period_aggregations['all']
engineered_features.head()



Ticker-level features shape [all]: (503, 48)

Features:
['Ticker', 'Company', 'Industrykey', 'Category', 'Return_mean', 'Return_std', 'Return_skew', 'Volume_accum_mean', 'Volume_accum_std', 'Volume_accum_max', 'MarketCap_mean', 'MarketCap_std', 'MarketCap_min', 'MarketCap_max', 'Volume_Change_mean', 'Volume_Change_std', 'EPS_Change_mean', 'EPS_Change_std', 'NetWorth_Change_mean', 'NetWorth_Change_std', 'NIL_Change_mean', 'NIL_Change_std', 'Volatility_2Q_mean', 'Volatility_2Q_max', 'Avg_Return_2Q_mean', 'Avg_Volume_Change_2Q_mean', 'EPS_Volatility_2Q_mean', 'EPS_Volatility_2Q_max', 'Avg_EPS_Change_2Q_mean', 'NetWorth_Volatility_2Q_mean', 'NetWorth_Volatility_2Q_max', 'Avg_NetWorth_Change_2Q_mean', 'NIL_Volatility_2Q_mean', 'NIL_Volatility_2Q_max', 'Avg_NIL_Change_2Q_mean', 'Volatility_4Q_mean', 'Volatility_4Q_max', 'Avg_Return_4Q_mean', 'Avg_Volume_Change_4Q_mean', 'EPS_Volatility_4Q_mean', 'EPS_Volatility_4Q_max', 'Avg_EPS_Change_4Q_mean', 'NetWorth_Volatility_4Q_mean', 'NetWorth_Vola

,Ticker,Company,Industrykey,Category,Return_mean,Return_std,Return_skew,Volume_accum_mean,Volume_accum_std,Volume_accum_max,...,Avg_Volume_Change_4Q_mean,EPS_Volatility_4Q_mean,EPS_Volatility_4Q_max,Avg_EPS_Change_4Q_mean,NetWorth_Volatility_4Q_mean,NetWorth_Volatility_4Q_max,Avg_NetWorth_Change_4Q_mean,NIL_Volatility_4Q_mean,NIL_Volatility_4Q_max,Avg_NIL_Change_4Q_mean
0,A,Agilent Technologies,diagnostics-research,sp500,0.043067,0.117935,-0.036898,4.063654e+07,9.984756e+06,7.083990e+07,...,0.049141,0.981866,3.123564,0.273767,0.037271,0.106995,0.003843,7.080844,28.442236,2.664870
1,AAPL,Apple Inc.,consumer-electronics,sp500,0.071759,0.156266,-0.070810,2.275765e+09,1.038062e+09,6.280072e+09,...,0.014215,0.542979,0.635845,0.133357,0.067775,0.138963,-0.013587,0.544057,0.631385,0.157407
2,ABBV,AbbVie,drug-manufacturers-general,sp500,0.053754,0.125594,0.308907,1.482559e+08,4.964051e+07,3.681825e+08,...,0.059068,1.042537,4.408229,0.440800,0.132883,0.464296,0.052540,1.818792,17.237456,0.794460
3,ABNB,Airbnb,travel-services,sp500,0.008785,0.198991,-0.249100,1.157236e+08,3.252025e+07,2.084769e+08,...,0.012426,7.451058,11.924797,3.378764,0.154001,0.322918,0.076989,5.874528,9.007148,2.637675
4,ABT,Abbott Laboratories,medical-devices,sp500,0.037560,0.097236,-0.021515,1.296637e+08,4.166209e+07,3.113584e+08,...,0.072598,1.016708,3.318321,0.444757,0.111448,0.497696,0.024302,1.018145,3.335739,0.441611


In [7]:
def feature_statistics(engineered_features, feature_cols):
    gig_work_stats = engineered_features[engineered_features['Category'] == 'gig-work'][feature_cols].mean()
    mlm_work_stats = engineered_features[engineered_features['Category'] == 'mlm-work'][feature_cols].mean()
    sp500_stats = engineered_features[(engineered_features['Category'] == 'sp500')][feature_cols].mean()
    return gig_work_stats, mlm_work_stats, sp500_stats


gig_work_stats, mlm_work_stats, sp500_stats = feature_statistics(engineered_features, feature_cols)
gig_work_stats_pre_covid, mlm_work_stats_pre_covid, sp500_stats_pre_covid = feature_statistics(pre_covid_ef, pre_covid_fc)
gig_work_stats_covid, mlm_work_stats_covid, sp500_stats_covid = feature_statistics(covid_ef, covid_fc)
gig_work_stats_post_covid, mlm_work_stats_post_covid, sp500_stats_post_covid = feature_statistics(post_covid_ef, post_covid_fc)


print("\nGig Work Feature Statistics:")
print("\nOverall:\n", gig_work_stats, "\n\nPre-COVID:\n", gig_work_stats_pre_covid, "\n\nCOVID:\n", gig_work_stats_covid, "\n\nPost-COVID:\n", gig_work_stats_post_covid)
print("\nMLM Work Feature Statistics:")
print("\nOverall:\n", mlm_work_stats, "\n\nPre-COVID:\n", mlm_work_stats_pre_covid, "\n\nCOVID:\n", mlm_work_stats_covid, "\n\nPost-COVID:\n", mlm_work_stats_post_covid)
print("\nS&P 500 Categories Feature Statistics (Mean):")
print("\nOverall:\n", sp500_stats, "\n\nPre-COVID:\n", sp500_stats_pre_covid, "\n\nCOVID:\n", sp500_stats_covid, "\n\nPost-COVID:\n", sp500_stats_post_covid)


Gig Work Feature Statistics:

Overall:
 Return_mean                    6.010165e-02
Return_std                     3.126608e-01
Return_skew                    8.148795e-01
Volume_accum_mean              1.576998e+08
Volume_accum_std               6.718336e+07
Volume_accum_max               3.359505e+08
MarketCap_mean                 5.039951e+10
MarketCap_std                  4.584605e+10
MarketCap_min                  2.943726e+09
MarketCap_max                  1.823447e+11
Volume_Change_mean             1.083943e-01
Volume_Change_std              5.066699e-01
EPS_Change_mean               -1.266351e-01
EPS_Change_std                 3.655750e+00
NetWorth_Change_mean           1.427594e-02
NetWorth_Change_std            1.702095e-01
NIL_Change_mean               -2.524363e-01
NIL_Change_std                 3.218192e+00
Volatility_2Q_mean             2.147945e-01
Volatility_2Q_max              7.942174e-01
Avg_Return_2Q_mean             5.775776e-02
Avg_Volume_Change_2Q_mean      1.15

In [8]:
# save the overall clustering data for feature selection and clustering steps
engineered_features.to_csv('../../data/clustering/engineered_features.csv', index=False)
engineered_features

# save the era data
pre_covid_ef.to_csv('../../data/clustering/pre_covid_engineered_features.csv', index=False)
covid_ef.to_csv('../../data/clustering/covid_engineered_features.csv', index=False)  
post_covid_ef.to_csv('../../data/clustering/post_covid_engineered_features.csv', index=False)
